In [61]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import json

data_loc = "../data/synthetic/components"

### Set up dropdowns

In [62]:
with open(f'{data_loc}/motherboards.json', 'r') as file:
    mobo_data = json.load(file)

mobo_choices = [(f"{mobo['brand']} {mobo['model']}", mobo['id']) for mobo in mobo_data]

In [63]:
# everyone but mobo has id + name
# function time baby

def get_component_choices(component):
    with open(f'{data_loc}/{component}.json', 'r') as file:
        component_data = json.load(file)
    return [(f"{item['name']}", item['id']) for item in component_data]

In [64]:
# also all components need the same stuff for the dropdowns, so let's make a function for that too

def create_dropdown(component, choices=None, display_it=False):
    choices = choices if choices is not None else get_component_choices(component)
    dropdown = widgets.Select(
        options=choices,
        value=choices[0][1],
        description=f'{component}:',
        disabled=False,
        layout=widgets.Layout(width='max-content'),
        rows=len(choices),
    )
    if display_it == True: display(dropdown)
    return dropdown

# note to future self: mobo_dropdown.value to get selected ID, mobo_dropdown.label to get selected label

In [65]:
mobo_dropdown = create_dropdown('motherboards', mobo_choices)

component_list = ["cases", "coolers", "cpus", "fans", "gpus", "lighting", "psus", "ram", "storage"]
dropdowns = [create_dropdown(component) for component in component_list]
dropdowns.insert(0, mobo_dropdown)

I want to check whether components are compatible. There are a couple of ways to do this. I could poll with every selection, or I can just provide a big fat button so you submit the build, then check. 

Performance wise, the latter wins, but I don't think any of this is performance intensive enough to warant a hack. 

Ok, gonna go for polling. 

I also think that gives more opportunity for learning.

### Compatibility checking


In [66]:
def get_mobo_name(mobo_data):
    return f"{mobo_data["brand"]} {mobo_data["model"]}"

# CASE COMPATIBILITY
def is_case_compatible(build):
    case_problems = {}
    case = build["cases"]
    # check that mobo form factor is supported
    mobo = build["motherboards"]
    if not mobo["form_factor"] in case["motherboard_form_factors"]:
        case_problems["motherboard_form_factors"] = (
            f"{get_mobo_name(mobo)} motherboard form factor ({mobo['form_factor']}) is not supported by the {case['name']} case."
        )
    
    # check that gpu length is supported
    gpu = build["gpus"]
    if gpu["length_mm"] > case["gpu_max_length_mm"]:
        case_problems["gpu_max_length_mm"] = (
            f"{gpu['name']} GPU length ({gpu['length_mm']} mm) exceeds the {case['name']} case limit ({case['gpu_max_length_mm']} mm)."
        )

    # check that cpu height is supported
    cooler = build["coolers"]
    if cooler["type"] == "air" and cooler["height_mm"] > case["cpu_cooler_max_height_mm"]:
        case_problems["cpu_cooler_max_height_mm"] = (
            f"{cooler['name']} cooler height ({cooler['height_mm']} mm) exceeds the {case['name']} case limit ({case['cpu_cooler_max_height_mm']} mm)."
        )

    # check that psu form factor is supported
    psu = build["psus"]
    if psu["form_factor"] not in case["supported_psu_form_factors"]:
        case_problems["supported_psu_form_factors"] = (
            f"{psu['name']} PSU form factor ({psu['form_factor']}) is not supported by the {case['name']} case."
        )

    # check that radiator size is supported
    if cooler["type"] == "liquid" and cooler["radiator_mm"] not in case["radiator_sizes_mm"]:
        case_problems["radiator_sizes_mm"] = (
            f"{cooler['name']} radiator size ({cooler['radiator_mm']} mm) is not supported by the {case['name']} case."
        )

    # check that sata 2.5 is supported
    storage = build["storage"]
    if storage["form_factor"] == "SATA_2.5" and case["sata_bays_2_5"] < 1:
        case_problems["sata_bays_2_5"] = (
            f"{case['name']} case does not support 2.5-inch SATA drives."
        )

    # check that sata 3.5 is supported
    if storage["form_factor"] == "SATA_3.5" and case["sata_bays_3_5"] < 1:
        case_problems["sata_bays_3_5"] = (
            f"{case['name']} case does not support 3.5-inch SATA drives."
        )
        
    return len(case_problems.keys()), case_problems

In [ ]:
# loads our entire catalog from the data folder
CATEGORIES = ['cases', 'coolers', 'cpus', 'fans', 'gpus', 'lighting', 'motherboards', 'psus', 'ram', 'storage']
CATALOG = {}
for category in CATEGORIES:
    with open(f'{data_loc}/{category}.json', 'r') as file:
        cat_data = json.load(file)
    CATALOG[category] = cat_data

# helper to grab all of a part's data from the catalog, looking up via ID.
def get_part_by_ID(component, id):
    return next((item for item in CATALOG[component] if item["id"] == id), None)

# I'll need a way to grab the current build
def get_current_build():
    # makes a dictionary with this shape:
    #   { <component type>: <component details from catalog>, ... }
    return { dropdown.description[:-1]: get_part_by_ID(dropdown.description[:-1], dropdown.value) for dropdown in dropdowns }

# checks if the current selections make a compatible build
case_output = widgets.Output()
def check_build(change):
    build = get_current_build()
    case_compat, case_issues = is_case_compatible(build)
    with case_output:
        clear_output(wait=True)
        # TODO format this nicer
        if case_compat:
            print(f"Case compatibility issues ({case_compat}):")
            for issue in case_issues.values():
                print(f"- {issue}")
        else:
            print("Case is compatible.")

### Set up observers for dropdown choices

In [68]:
# add an observer for every dropdown
def update_build_text(change):
    build_text.value = f"{newline.join([dropdown.value for dropdown in dropdowns])}"

for dropdown in dropdowns:
    dropdown.observe(update_build_text, names="value")
    dropdown.observe(check_build, names="value")

### Display

In [69]:
newline = "\n"
build_text = widgets.Textarea(
    value=f"{newline.join([dropdown.value for dropdown in dropdowns])}",
    placeholder='Type something',
    description='Build:',
    disabled=True,
    rows=len(dropdowns),
    layout=widgets.Layout(justify_content='flex-end', width='max-content')
)

In [70]:
widgets.GridBox(
    children=dropdowns + [build_text, case_output],
    layout=widgets.Layout(
        # width='50%',
        grid_template_columns='auto auto auto',
        grid_template_rows='auto auto auto auto', 
        grid_gap='5px 10px',
    )
)

GridBox(children=(Select(description='motherboards:', layout=Layout(width='max-content'), options=(('Fictional…